In [1]:
!pip install pyscipopt pyscipopt-ml -q

# Постановка задачи

(этот пример основан на примере от авторов фреймворка -> Алексей Никоноров -> Максим Гончаров)

Производитель автомобилей хочет определить технические параметры новой машины так, чтобы максимизировать доход от продаж. При этом производитель хочет, чтобы эта машина была малогабаритная: $\text{Length} \cdot \text{Width} \leq \text{MinSize}$.  

У нас есть датасет с историей продаж других машин: объемы продаж, цены и характеристики. Среди характеристик машин есть непрерывные и дискретные. Мы обучили модель, чтобы прогнозировать объём продаж машин от их параметров и цены:  

$$\begin{aligned}
&\text{Sales}(f_1, ..., f_n, p) \ \colon \mathbb{R}^{n+1} \to \mathbb{R}
\end{aligned}$$

* $f_1, ..., f_n$ - параметры машины
* $p$ - цена

_Кросс-эффекты по цене не учитываются_

Оптимизационная задача формулируется так:  

\begin{aligned}
&\textrm{maximize}  \quad  p \cdot s  & \\
&\textrm{subject to} & \\ 
                          & \qquad s = \text{Sales}(f_1, ..., f_n, p)  & \ \\
                          & \qquad f_i \cdot f_j <= \text{MinSize} & \text{if } i = \text{Length} \ \land \ j = \text{Width}\\
                          & \qquad p,s \in \mathbb{R} & \ \\
                          & \qquad f_i \in \mathbb{R} & \text{if } i \neq \text{VehicleType} \\
                          & \qquad f_i \in \{0, 1\} & \text{if } i = \text{VehicleType} \\
\end{aligned}

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Сначала импортируйте torch
import torch
import pandas as pd
import numpy as np

# Затем SCIP
from pyscipopt import Model
from pyscipopt_ml.add_predictor import add_predictor_constr

## Подготовка данных для обучения прогнозной модели

In [2]:
# Читаем датасет
df = pd.read_csv('car_sales.csv')
print(df)

    Manufacturer    Model  Sales_in_thousands  __year_resale_value  \
0          Acura  Integra              16.919               16.360   
1          Acura       TL              39.384               19.875   
2          Acura       CL              14.114               18.225   
3          Acura       RL               8.588               29.725   
4           Audi       A4              20.397               22.255   
..           ...      ...                 ...                  ...   
151        Volvo      V40               3.545               17.200   
152        Volvo      S70              15.245               20.600   
153        Volvo      V70              17.531               20.800   
154        Volvo      C70               3.493               31.000   
155        Volvo      S80              18.969               28.700   

    Vehicle_type  Price_in_thousands  Engine_size  Horsepower  Wheelbase  \
0      Passenger               21.50          1.8         140      101.2   
1      

In [3]:
# Будем использовать эти фичи -- характиристики машин
features = [
    'Vehicle_type',
    'Engine_size',
    'Horsepower',
    'Wheelbase',
    'Width',
    'Length',
    'Curb_weight',
    'Fuel_capacity',
    'Fuel_efficiency',
    'Power_perf_factor',
]

## Прогнозная модель -- pytorch

In [4]:
# Готовим датасет для обучения
# target
sales = torch.tensor(df['Sales_in_thousands'], dtype=torch.float32)
# predictors = features + price
df1 = df[features + ['Price_in_thousands']].copy()
# Дискретная переменная 1: 'Passenger', 0: 'Car'
df1['Vehicle_type'] = (df1['Vehicle_type'] == 'Passenger')
df1['Vehicle_type'] = df1['Vehicle_type'].astype("float")
# print(df1.dtypes)
X = torch.tensor(df1.to_numpy(), dtype=torch.float32)
# print(X)
print(X.shape)

torch.Size([156, 11])


In [5]:
# Создаём модель прогнозирования продаж (torch)
n_inputs = X.shape[1]
layers_sizes = (20, 20, 10)
reg_sales = torch.nn.Sequential(
            torch.nn.Linear(n_inputs, layers_sizes[0]),
            torch.nn.Sigmoid(),
            torch.nn.Linear(layers_sizes[0], layers_sizes[1]),
            torch.nn.Sigmoid(),
            torch.nn.Linear(layers_sizes[1], layers_sizes[2]),
            torch.nn.ReLU(),
            torch.nn.Linear(layers_sizes[2], 1),
        )
# print(reg_sales(X))

In [6]:
# Параметры обучения модели прогнозирования продаж (torch)
batch_size = X.shape[0]
n_epochs = 2000
n_train = X.shape[0]
n_batches = n_train // batch_size

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(reg_sales.parameters(), lr=0.001, weight_decay=0.0001)

In [7]:
epoch = 0
for epoch in range(n_epochs):
    # print(f"epoch: {epoch}")
    t_loss = 0
    i = 0
    for batch_num in range(n_batches):
        # print(f"batch_num: {batch_num}")
        batch_X = X[batch_num * batch_size: (batch_num + 1) * batch_size, :]
        batch_sales = sales[batch_num * batch_size: (batch_num + 1) * batch_size].flatten()

        sales_pred = reg_sales(batch_X).flatten()
        loss = criterion(sales_pred, batch_sales)
        i += 1
        t_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(t_loss / i)

print(f"finished: {criterion(sales, reg_sales(batch_X).flatten()).item()}")

7435.16650390625
7433.64892578125
7432.1435546875
7430.65869140625
7429.1923828125
7427.7451171875
7426.3125
7424.89599609375
7423.4951171875
7422.1103515625
7420.74365234375
7419.39111328125
7418.0546875
7416.7314453125
7415.4189453125
7414.1171875
7412.822265625
7411.5322265625
7410.24658203125
7408.96337890625
7407.68212890625
7406.39990234375
7405.115234375
7403.8310546875
7402.54345703125
7401.25146484375
7399.95654296875
7398.65771484375
7397.35400390625
7396.0439453125
7394.72900390625
7393.40869140625
7392.08154296875
7390.74658203125
7389.45361328125
7388.34033203125
7387.34228515625
7386.32861328125
7385.294921875
7384.2421875
7383.17138671875
7382.08642578125
7380.98388671875
7379.86279296875
7378.71728515625
7377.54150390625
7376.33642578125
7375.1025390625
7373.8427734375
7372.5546875
7371.2275390625
7369.85888671875
7368.44482421875
7366.982421875
7365.4853515625
7363.9833984375
7362.48095703125
7360.97412109375
7359.455078125
7357.91650390625
7356.3486328125
7354.7353515

## Обучение прогнозной модели закончилось -- приступаем к формированию оптимизационной модели

In [8]:
model = Model()
model.redirectOutput()

# Цена нашего продукта
# Накладываем ограничения на исторические ценовые диапазоны
lb_price = df['Price_in_thousands'].quantile(0.1)
ub_price = df['Price_in_thousands'].quantile(0.9)
x_price = model.addVar(vtype='C', lb=lb_price, ub=ub_price, name='price')

# Продажи нашего продукта
# Накладываем ограничения на исторические диапазоны объёмов продаж -- обрезаем экстемальные прогнозы модели
lb_sales = df['Sales_in_thousands'].quantile(0.1)
ub_sales = df['Sales_in_thousands'].quantile(0.9)
x_sales = model.addVar(vtype='C', lb=lb_sales, ub=ub_sales, name='sales')

# Выручка
# В SCIP нельзя задавать нелинейные целевые функции,
# поэтому используем трюк с введением доп. переменной
x_revenue = model.addVar(vtype='C', lb=lb_price * lb_sales, ub=ub_price * ub_sales, name='revenue')
model.setObjective(x_revenue, 'maximize')
model.addCons(x_revenue <= x_price * x_sales, name='revenue')

# Создаём переменные решения для нашего продукта, соответствующие его фичам
x_features = []
for feature in features:
    if feature == 'Vehicle_type':
        x_features.append(model.addVar(vtype='B', name=feature))
    else:
        lb = df[feature].quantile(0.1)
        ub = df[feature].quantile(0.9)
        x_features.append(model.addVar(vtype='C', lb=lb, ub=ub, name=feature))

# Мы хотим ограничить по размеру новую машину
i_width = features.index('Width')
i_length = features.index('Length')
max_size = (df['Width'] * df['Length']).quantile(0.3)
model.addCons(x_features[i_width] * x_features[i_length] <= max_size, name='size')

"""
Создаём специальное ограничение, которое связывает целевую переменную ML-модели (прогноз продаж) с целевой переменной решения оптимизационной задачи
через переменные решения входных фичей
"""
# Ограничения, которые определяют переменную x_sales
add_predictor_constr(
    model, # оптимизационная модель
    reg_sales, # прогнозная модель
    np.array(x_features + [x_price]).reshape((1, -1)), # переменные решения, соотв. предикторам
    np.array([[x_sales]]), # переменная решения, соотв. прогнозу
    epsilon=0.0001,
    unique_naming_prefix='model_sales_',
)

In [9]:
print('=== Наши переменные ===')
old_vars = []
for v in model.getVars():
    if 'model_' not in str(v):
        old_vars.append(v)
        print(v, v.vtype())
print(len(old_vars))

=== Наши переменные ===
Vehicle_type BINARY
sales CONTINUOUS
revenue CONTINUOUS
price CONTINUOUS
Engine_size CONTINUOUS
Horsepower CONTINUOUS
Wheelbase CONTINUOUS
Width CONTINUOUS
Length CONTINUOUS
Curb_weight CONTINUOUS
Fuel_capacity CONTINUOUS
Fuel_efficiency CONTINUOUS
Power_perf_factor CONTINUOUS
13


In [10]:
print('=== Переменные, которые добавил фреймворк ===')
new_vars = []
for v in model.getVars():
    if 'model_' in str(v):
        new_vars.append(v)
        print(v, v.vtype())
print(len(new_vars))

=== Переменные, которые добавил фреймворк ===
model_sales_layer_0_0_0 CONTINUOUS
model_sales_layer_0_0_1 CONTINUOUS
model_sales_layer_0_0_2 CONTINUOUS
model_sales_layer_0_0_3 CONTINUOUS
model_sales_layer_0_0_4 CONTINUOUS
model_sales_layer_0_0_5 CONTINUOUS
model_sales_layer_0_0_6 CONTINUOUS
model_sales_layer_0_0_7 CONTINUOUS
model_sales_layer_0_0_8 CONTINUOUS
model_sales_layer_0_0_9 CONTINUOUS
model_sales_layer_0_0_10 CONTINUOUS
model_sales_layer_0_0_11 CONTINUOUS
model_sales_layer_0_0_12 CONTINUOUS
model_sales_layer_0_0_13 CONTINUOUS
model_sales_layer_0_0_14 CONTINUOUS
model_sales_layer_0_0_15 CONTINUOUS
model_sales_layer_0_0_16 CONTINUOUS
model_sales_layer_0_0_17 CONTINUOUS
model_sales_layer_0_0_18 CONTINUOUS
model_sales_layer_0_0_19 CONTINUOUS
model_sales_layer_1_0_0 CONTINUOUS
model_sales_layer_1_0_1 CONTINUOUS
model_sales_layer_1_0_2 CONTINUOUS
model_sales_layer_1_0_3 CONTINUOUS
model_sales_layer_1_0_4 CONTINUOUS
model_sales_layer_1_0_5 CONTINUOUS
model_sales_layer_1_0_6 CONTINUOUS

In [11]:
model.optimize()

presolving:
(round 1, fast)       18 del vars, 10 del conss, 0 add conss, 15 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 2, fast)       18 del vars, 10 del conss, 0 add conss, 94 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 3, medium)     18 del vars, 18 del conss, 0 add conss, 94 chg bounds, 0 chg sides, 8 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
   (0.0s) probing cycle finished: starting next cycle
   (0.0s) symmetry computation started: requiring (bin +, int +, cont +), (fixed: bin -, int -, cont -)
   (0.0s) no symmetry present (symcode time: 0.00)
presolving (4 rounds: 4 fast, 2 medium, 1 exhaustive):
 18 deleted vars, 18 deleted constraints, 0 added constraints, 94 tightened bounds, 0 added holes, 0 changed sides, 8 changed coefficients
 7 implications, 0 cliques, 0 implied integral variables (0 bin, 0 int, 0 cont)
presolved problem has 105 variables (1 bin, 0 int, 104 cont) and 95

In [ ]:
df1 = pd.DataFrame(data={
    'revenue': df['Sales_in_thousands'] * df['Price_in_thousands'],
    'sales': df['Sales_in_thousands'],
    'price': df['Price_in_thousands'],
    'size': df['Width'] * df['Length'],
})
df1 = df1.sort_values(['revenue'], ascending=False)

entries = []
for i, feature in enumerate(features):
    if feature == 'Vehicle_type':
        label = ['Car', 'Passenger'][int(round(model.getVal(x_features[i])))]
        entries.append((feature, label, df[feature].min(), df[feature].max()))
    else:
        entries.append((feature, model.getVal(x_features[i]), df[feature].quantile(0.1), df[feature].quantile(0.9)))
our_features = pd.DataFrame(data=entries, columns=['feature', 'value', 'dataset_lb', 'dataset_ub'])
sales = model.getVal(x_sales)
price = model.getVal(x_price)
revenue = model.getVal(x_sales) * model.getVal(x_price)
size = model.getVal(x_features[features.index('Width')] * x_features[features.index('Length')])

print('=== Топ 10 продаж из датасета ===')
print(df1.head(10))
print()
print('=== Ожидаемые продажи нашей машины ===')
print(f'revenue {revenue:.2f}, sales {sales:.2f}, price {price:.2f}, size {size:.2f}')
print()
print('=== Параметры нашей машины ===')
print(our_features)